# Phase 4: answering from the user's documents

Phase 4 lets a research run read the user's own documents. A PDF is ingested once (read by `pdf-parser-mcp`, cut into chunks that are exact slices of their pages, embedded, stored per document); a sub-question routed to the documents is answered by the Self-RAG subgraph (a passage shaped like the answer searched beside the question, hybrid retrieval, grading, support and usefulness checks); and the answer becomes findings cited by page, beside the web findings.

This notebook keeps the record of the phase:

1. **Part A, ingestion measured:** the annual report ingested on a clean run into a temporary store, every embedding request timed, to find where the time goes (the gap left open in concept `document-ingestion`: about 2 s a request, above the limiter's 1.25 s).
2. **Part B, a brief from the web and the annual report:** the same topic as Phases 1 to 3, now with Symbotic's annual report attached; the brief is rendered below and saved to `notebooks/phase_04_brief.md`, with the run numbers and a comparison with the earlier phases.
3. **The phase record:** the problems met while opening, building, and closing Phase 4, and what was decided.

**Before running:** start both servers (`.\scripts\start_mcp_servers.ps1`), and have the annual report at `data/uploads/symbotic-annual-report.pdf`, ingested once (`tests/live/test_ingestion_live.py`). Part A uses a temporary store, so the real one is not touched. The whole notebook costs about 35 embedding requests, up to 6 Tavily searches, and a few dozen chat requests.

In [ ]:
# Work from the project root, wherever Jupyter started: settings.py reads .env
# from the working folder, and the graph package is imported from the root.
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT.name}")

In [ ]:
import logging
import re
import shutil
import socket
import statistics
import tempfile
import time
from itertools import pairwise
from urllib.parse import urlsplit
from uuid import uuid4

from IPython.display import Markdown, display
from langchain_core.embeddings import Embeddings
from langchain_core.tracers.langchain import wait_for_all_tracers

from app.ingestion import IngestionService, file_fingerprint
from graph.document_store import DocumentStore
from graph.llm import (
    EMBED_REQUESTS_IN_FLIGHT,
    MODEL_CALL_BUDGET_KEY,
    ModelGateway,
    build_mistral_embeddings,
)
from graph.mcp_client import START_SERVER_COMMAND, McpToolbox
from graph.nodes.researcher import PAGES_PER_QUESTION, SEARCH_RESULTS_PER_QUESTION
from graph.prompts import planner as planner_prompt
from graph.prompts import self_rag as self_rag_prompts
from graph.prompts import web_findings as web_findings_prompt
from graph.prompts import writer as writer_prompts
from graph.report import cited_finding_ids, render_brief
from graph.run_budget import planned_request_budget
from graph.supervisor import build_graph, research_run_config
from settings import settings

ANNUAL_REPORT_PATH = PROJECT_ROOT / "data" / "uploads" / "symbotic-annual-report.pdf"


class RunLogRecorder(logging.Handler):
    # Keeps the graph's log messages, so skips, drops, retries, and timings can be read.

    def __init__(self) -> None:
        super().__init__(level=logging.INFO)
        self.messages: list[str] = []

    def emit(self, record: logging.LogRecord) -> None:
        self.messages.append(record.getMessage())


# Every module under graph/ and app/ logs to a child of these loggers.
run_log = RunLogRecorder()
for logger_name in ("graph", "app"):
    package_logger = logging.getLogger(logger_name)
    package_logger.setLevel(logging.INFO)
    for old_recorder in [h for h in package_logger.handlers if isinstance(h, RunLogRecorder)]:
        package_logger.removeHandler(old_recorder)  # re-running this cell must not count twice
    package_logger.addHandler(run_log)

In [ ]:
# Both servers must be listening before any request is spent.
def server_is_listening(server_url: str) -> bool:
    address = urlsplit(server_url)
    try:
        with socket.create_connection((address.hostname, address.port or 80), 1.0):
            return True
    except OSError:
        return False


for server_name, server_url in (
    ("search-mcp", settings.vre_search_mcp_url),
    ("pdf-parser-mcp", settings.vre_pdf_parser_mcp_url),
):
    if not server_is_listening(server_url):
        raise RuntimeError(f"{server_name} is not running; start it with: {START_SERVER_COMMAND}")
    print(f"{server_name} is running at {server_url}")

stored_report = DocumentStore().find_by_fingerprint(file_fingerprint(ANNUAL_REPORT_PATH))
if stored_report is None:
    raise RuntimeError("Ingest the annual report first: tests/live/test_ingestion_live.py")
print(
    f"Annual report stored as {stored_report.document_id}: "
    f"{stored_report.page_count} pages, {stored_report.chunk_count} chunks"
)

## Part A: where ingestion spends its time

Concept `document-ingestion` measured 87 s for the annual report with three embedding batches in flight (D-49), about 2 s a request, while the limiter alone allows one every 1.25 s (`VRE_EMBED_RPS` 0.8). The log lines had no timestamps, so where the rest went was not known.

Here the report is ingested again into a **temporary store** (the real one is not touched, and nothing is skipped as already stored), with every embedding request timed where it leaves the gateway: when it was sent, after the limiter let it go, and when its reply arrived. This wrapper is a measuring tool for this notebook only; the product code is unchanged.

In [ ]:
class TimedEmbeddings(Embeddings):
    # Times each request to the real embedding model; adds nothing else.

    def __init__(self, inner: Embeddings) -> None:
        self.inner = inner
        self.requests: list[tuple[float, float, int]] = []  # (sent, replied, texts)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self.inner.embed_documents(texts)

    def embed_query(self, text: str) -> list[float]:
        return self.inner.embed_query(text)

    async def aembed_documents(self, texts: list[str]) -> list[list[float]]:
        sent_at = time.perf_counter()
        vectors = await self.inner.aembed_documents(texts)
        self.requests.append((sent_at, time.perf_counter(), len(texts)))
        return vectors


timed_embeddings = TimedEmbeddings(build_mistral_embeddings(settings.vre_embed_model))
measuring_gateway = ModelGateway(embedding_model=timed_embeddings)
temporary_store_folder = Path(tempfile.mkdtemp(prefix="vre-ingestion-"))
run_log.messages.clear()

ingestion_started_at = time.perf_counter()
try:
    ingestion_result = await IngestionService(
        measuring_gateway, DocumentStore(temporary_store_folder)
    ).ingest(ANNUAL_REPORT_PATH)
finally:
    shutil.rmtree(temporary_store_folder, ignore_errors=True)
ingestion_seconds = time.perf_counter() - ingestion_started_at

assert not ingestion_result.already_stored, "the temporary store should start empty"
print(
    f"Ingested {ingestion_result.document.page_count} pages, "
    f"{ingestion_result.document.chunk_count} chunks in {ingestion_seconds:.1f} s"
)

In [ ]:
requests_timed = sorted(timed_embeddings.requests)
first_sent_at = requests_timed[0][0]
last_replied_at = max(replied for _, replied, _ in requests_timed)
durations = [replied - sent for sent, replied, _ in requests_timed]
gaps_between_sends = [later[0] - earlier[0] for earlier, later in pairwise(requests_timed)]
texts_per_request = [text_count for _, _, text_count in requests_timed]
embedding_seconds = last_replied_at - first_sent_at
before_first_request = first_sent_at - ingestion_started_at  # reading pages and chunking


def in_flight_at(moment: float) -> int:
    return sum(sent <= moment < replied for sent, replied, _ in requests_timed)


average_in_flight = statistics.mean(in_flight_at(sent) for sent, _, _ in requests_timed)
limiter_floor = 1 / settings.vre_embed_rps
concurrency_floor = statistics.mean(durations) / EMBED_REQUESTS_IN_FLIGHT

ingestion_numbers = {
    "total seconds": round(ingestion_seconds, 1),
    "before the first request (pages, chunks)": round(before_first_request, 1),
    "embedding seconds (first sent to last reply)": round(embedding_seconds, 1),
    "requests": len(requests_timed),
    "texts per request": f"{min(texts_per_request)} to {max(texts_per_request)}",
    "seconds per request, overall": round(embedding_seconds / len(requests_timed), 2),
    "reply time, median": round(statistics.median(durations), 2),
    "reply time, slowest": round(max(durations), 2),
    "gap between sends, median": round(statistics.median(gaps_between_sends), 2),
    "requests in flight when one is sent, average": round(average_in_flight, 2),
    "limiter floor (1 / VRE_EMBED_RPS)": round(limiter_floor, 2),
    f"in-flight floor (mean reply time / {EMBED_REQUESTS_IN_FLIGHT})": round(concurrency_floor, 2),
    "transient retries": sum("transient error" in m for m in run_log.messages),
}
for measure, value in ingestion_numbers.items():
    print(f"{measure:<48}{value}")

# The pace is set by whichever floor is higher: the limiter's spacing, or how
# long replies take divided by how many may wait at once.
bound_by = "the limiter" if limiter_floor >= concurrency_floor else "reply time with 3 in flight"
print(f"\nThe embedding pace is bound by {bound_by}.")

## Part B: a brief from the web and the annual report

The same topic as Phases 1 to 3, with the annual report attached. The planner sees the file name and routes each sub-question to the web, the document, or both; web and document branches run side by side; the run's request budget is sized to the plan (D-58).

In [ ]:
TOPIC = "Due diligence on Symbotic: business model, customers, and main risks"

# CONCEPT: tracing-from-day-one. The run id is chosen before the run, so the
# trace can be looked up afterwards; the config carries the run's budget.
run_id = uuid4()
run_config = research_run_config(TOPIC, run_id)
run_log.messages.clear()

# CONCEPT: parallel-fan-out. "updates" yields each node's result as it
# finishes; "values" yields the whole state after each step.
BRANCH_NODES = ("web_researcher", "document_researcher")
finished_at_by_node: dict[str, float] = {}
branch_finished_at: list[tuple[str, str, float]] = []
final_state: dict = {}
started_at = time.perf_counter()

async with McpToolbox.connect(settings.vre_search_mcp_url) as toolbox:
    graph = build_graph(ModelGateway(), toolbox)
    async for stream_mode, chunk in graph.astream(
        {"topic": TOPIC, "document_ids": [stored_report.document_id]},
        config=run_config,
        stream_mode=["updates", "values"],
    ):
        if stream_mode == "values":
            final_state = chunk
            continue
        seconds_so_far = time.perf_counter() - started_at
        for node_name, node_update in chunk.items():
            finished_at_by_node[node_name] = seconds_so_far
            if node_name in BRANCH_NODES and node_update["findings"]:
                sub_question_id = node_update["findings"][0].sub_question_id
                branch_finished_at.append((node_name, sub_question_id, seconds_so_far))

elapsed_seconds = time.perf_counter() - started_at
print(f"Done in {elapsed_seconds:.1f} s")
print("Plan:")
for sub_question in final_state["sub_questions"]:
    print(f"  {sub_question.id} [{sub_question.route}] {sub_question.text}")

In [ ]:
brief_markdown = render_brief(TOPIC, final_state["draft"], final_state["findings"])
display(Markdown(brief_markdown))

In [ ]:
def count_messages(fragment: str) -> int:
    return sum(fragment in message for message in run_log.messages)


def dropped_by_reason() -> dict[str, int]:
    reasons = re.findall(r"dropped a finding \(([^)]+)\)", "\n".join(run_log.messages))
    return {reason: reasons.count(reason) for reason in sorted(set(reasons))}


findings = final_state["findings"]
sub_questions = final_state["sub_questions"]
kept_ids = {finding.id for finding in findings if finding.found}
call_budget = run_config["configurable"][MODEL_CALL_BUDGET_KEY]
planner_seconds = finished_at_by_node["planner"]
research_seconds = finished_at_by_node["research_check"] - planner_seconds
writer_seconds = elapsed_seconds - finished_at_by_node["research_check"]
cited_ids = cited_finding_ids(final_state["draft"]) & kept_ids

run_numbers = {
    "sub-questions": len(sub_questions),
    "routes": ", ".join(q.route for q in sub_questions),
    "branches run": len(branch_finished_at),
    "web findings kept": sum(f.found for f in findings if f.source_type == "web"),
    "document findings kept": sum(f.found for f in findings if f.source_type == "document"),
    "sub-questions with no evidence": sum(not finding.found for finding in findings),
    "findings dropped by the checks": sum(dropped_by_reason().values()),
    "pages skipped": count_messages(": skipped "),
    "duplicate pages skipped (D-59)": count_messages("the same page as an earlier hit"),
    "citation fallbacks (D-57, choice A)": count_messages("no IsSUP quote found"),
    "sources cited": len(cited_ids),
    "document sources cited": sum(
        1 for f in findings if f.id in cited_ids and f.source_type == "document"
    ),
    "sections": len(final_state["draft"]),
    "schema retries": count_messages("did not fit"),
    "transient retries": count_messages("transient error"),
    "model requests": f"{call_budget.requests_sent} of {call_budget.limit} allowed",
    "planned budget (D-58)": planned_request_budget(sub_questions),
    "planner seconds": round(planner_seconds, 1),
    "research seconds": round(research_seconds, 1),
    "writer seconds": round(writer_seconds, 1),
    "seconds": round(elapsed_seconds, 1),
}
for measure, value in run_numbers.items():
    print(f"{measure:<40}{value}")
print(f"{'dropped, by reason':<40}{dropped_by_reason() or 'none'}")
print("\nDocument branches:")
for message in run_log.messages:
    if ": document research " in message:
        print(f"  {message}")
print("\nBranches finished, seconds after the planner:")
print(
    "  "
    + ", ".join(
        f"{qid} {node.split('_')[0]} {at - planner_seconds:.1f}"
        for node, qid, at in branch_finished_at
    )
)

In [ ]:
# Every document citation must be an exact slice of a stored chunk on its page.
locator_pattern = re.compile(r"^(?P<document_id>[^#]+)#page=(?P<page>\d+)$")
chunks_by_page: dict[int, list[str]] = {}
for stored_chunk in DocumentStore().chunks(stored_report.document_id):
    chunks_by_page.setdefault(stored_chunk.page, []).append(stored_chunk.text)

document_citations = [f for f in findings if f.found and f.source_type == "document"]
not_exact = [
    finding.id
    for finding in document_citations
    if not (
        (locator := locator_pattern.match(finding.locator))
        and any(
            text.startswith(finding.passage)
            for text in chunks_by_page.get(int(locator["page"]), [])
        )
    )
]
print(f"Document findings checked: {len(document_citations)}")
print(f"Not an exact slice of their page: {not_exact or 'none'}")

In [ ]:
# CONCEPT: tracing-from-day-one. Wait for the background uploads, then find
# this run by its id.
wait_for_all_tracers()

if settings.langsmith_tracing:
    from langsmith import Client

    langsmith_client = Client()
    for lookup_attempt in range(1, 6):
        try:
            traced_run = langsmith_client.read_run(run_id)
            trace_url = langsmith_client.get_run_url(
                run=traced_run, project_name=settings.langsmith_project
            )
            print(f"Trace: {trace_url}")
            break
        except Exception:  # a new run can take a few seconds to appear
            time.sleep(2 * lookup_attempt)
    else:
        print(f"Trace not found yet: search project {settings.langsmith_project} for {run_id}")
else:
    print("Tracing is off (LANGSMITH_TRACING=false)")

In [ ]:
# The committed record of the run. Only the brief and its provenance are
# written: never settings, keys, or LangSmith ids.
self_rag_versions = ", ".join(
    f"{prompt_id} {version}"
    for prompt_id, version in (
        self_rag_prompts.SEARCH_PASSAGE_PROMPT,
        self_rag_prompts.GRADING_PROMPT,
        self_rag_prompts.ANSWER_PROMPT,
        self_rag_prompts.SUPPORT_PROMPT,
        self_rag_prompts.REVISION_PROMPT,
        self_rag_prompts.USEFULNESS_PROMPT,
    )
)
prompt_versions = (
    f"P-01 {planner_prompt.VERSION}, P-02 {web_findings_prompt.VERSION}, "
    f"P-03 {writer_prompts.OUTLINE_VERSION}, P-04 {writer_prompts.SECTION_VERSION}; "
    f"Self-RAG {self_rag_versions}"
)
provenance_lines = [
    "> - Generated by `notebooks/phase_04.ipynb`, Phase 4. Same topic as "
    "`phase_01_brief.md` to `phase_03_brief.md`, now with Symbotic's annual report attached.",
    f"> - Document: `{ANNUAL_REPORT_PATH.name}`, {stored_report.page_count} pages, "
    f"{stored_report.chunk_count} chunks; document sources are cited by page.",
    f"> - Prompts: {prompt_versions}.",
    f"> - Models: `{settings.vre_chat_model}` for every tier (D-55), "
    f"at {settings.vre_decision_rps} requests a second through one limiter.",
    f"> - Web: `search-mcp` (MCP), {SEARCH_RESULTS_PER_QUESTION} results and up to "
    f"{PAGES_PER_QUESTION} pages per sub-question. Documents: the Self-RAG subgraph, "
    "a passage shaped like the answer searched beside the question (D-56).",
    f"> - Run: {run_numbers['sub-questions']} sub-questions ({run_numbers['routes']}), "
    f"{run_numbers['web findings kept']} web and {run_numbers['document findings kept']} "
    f"document findings kept, {run_numbers['sections']} sections, "
    f"{run_numbers['model requests']} model requests (the budget sized to this plan, D-58), "
    f"{run_numbers['seconds']} s.",
    "> - Every web passage occurs word for word on the page it cites, and every document "
    "passage is an exact slice of the page it cites, both checked in code. Whether each "
    "sentence is supported by its passage is checked from phase 5.",
]
title_line, brief_body = brief_markdown.split("\n", 1)
brief_file_text = "\n".join([title_line, "", *provenance_lines, brief_body])

brief_path = PROJECT_ROOT / "notebooks" / "phase_04_brief.md"
brief_path.write_text(brief_file_text, encoding="utf-8")
print(f"Wrote {brief_path.relative_to(PROJECT_ROOT)} ({len(brief_file_text.splitlines())} lines)")

In [ ]:
# Phases 1 to 3 as measured on the runs that closed them (Phase 3 on the
# Ministral models, before D-55); Phase 4 is this run.
comparison_rows = [
    ("Evidence from", "The web", "The web", "The web", "The web and the annual report"),
    (
        "Where a passage comes from",
        "Search snippet",
        "Checked copy from a page read",
        "Checked copy from a page read",
        "Checked copy, or an exact slice of a document page",
    ),
    (
        "Sub-questions researched",
        "One after another",
        "One after another",
        "All at once",
        f"All at once ({run_numbers['branches run']} branches, web and document)",
    ),
    (
        "Model and rate",
        "One throttle, 1 a second",
        "One throttle, 1 a second",
        "Two models, 2.5 and 0.4 a second",
        f"One model, {settings.vre_decision_rps} a second (D-55)",
    ),
    (
        "Findings kept",
        "18",
        "14",
        "14",
        f"{run_numbers['web findings kept']} web, {run_numbers['document findings kept']} document",
    ),
    ("Model requests", "", "", "16 of 30", run_numbers["model requests"]),
    ("Research stage", "", "", "31.0 s", f"{run_numbers['research seconds']} s"),
    ("Writer stage", "", "", "40.4 s", f"{run_numbers['writer seconds']} s"),
    ("Total time", "43.6 s", "99.4 s", "84.6 s", f"{run_numbers['seconds']} s"),
]
table_lines = [
    "| | Phase 1 | Phase 2 | Phase 3 | **Phase 4 (this run)** |",
    "| --- | --- | --- | --- | --- |",
    *(
        f"| {measure} | {first} | {second} | {third} | **{fourth}** |"
        for measure, first, second, third, fourth in comparison_rows
    ),
]
display(Markdown("\n".join(table_lines)))

## Problems we hit and what we decided

The phase record. One entry per problem met while opening, building, and closing Phase 4, each named by its phase and concept: what happened, what caused it (or what a check showed), the options, and the decision. Decision numbers refer to the project's decision log; every entry also has its own entry in `TROUBLESHOOTING.md` or the concept's notes.

### 1. The phase's tests needed code the design placed in Phase 8 (Phase 4, opening; D-44)

**What happened.** TC-DOC-03, TC-DOC-06, TC-RES-06, and TC-RES-10 need documents already chunked, embedded, and stored, but the design built ingestion with the upload API in Phase 8.

**Options.** Build ingestion now as plain code; keep it in Phase 8 and fill the store with a throwaway script; build the upload endpoint now too.

**Decision.** Ingestion became a Phase 4 concept, `document-ingestion`, as plain code; Phase 8 adds only the endpoint in front of it. The same kind of gap as Phase 2's P-02 (D-30).

### 2. A downloaded start script was blocked by Windows (Phase 4, concept `pdf-parser-mcp`)

**What happened.** `scripts/start_mcp_servers.ps1` was refused as "not digitally signed".

**Cause.** Files downloaded by a browser carry a mark of where they came from, and the execution policy refuses such scripts.

**Decision.** `take.ps1` unblocks every file it places, so this cannot recur for any delivered file.

### 3. A chunker bug the property tests first missed (Phase 4, concept `document-ingestion`)

**What happened.** A page with one word longer than a chunk broke the chunker, and the property tests passed, because their generator never produced such a word. One assertion could also never fail.

**Decision.** The generator now makes words of every length, including longer than a chunk; the assertion was rewritten to test what it claimed. Lesson kept: a property test is only as good as the inputs it can generate.

### 4. Servers, progress, and test order (Phase 4, concept `document-ingestion`)

**What happened.** The start script returned before the servers were listening, so the first request failed; the first ingestion of the annual report printed nothing for over a minute and was stopped by hand; two parallel tests relied on the order replies arrived in.

**Decision.** The start script waits until each server listens (up to 60 s); each finished embedding batch logs a progress line (D-49); parallel tests key scripted failures to their texts, never to arrival order.

### 5. Ingestion took 101 s; three requests in flight (Phase 4, concept `document-ingestion`; D-49)

**What happened.** The annual report (124 pages, 1,119 chunks, 35 embedding requests) took 101 s with one request at a time.

**Options.** One request at a time; several in flight under the same limiter; the provider's batch API (cheaper, but minutes to hours); a faster embedding model.

**Decision.** Up to three requests wait for their replies at once, under the same limiter, so the rate is unchanged: 87 s. About 2 s a request remained unexplained, above the limiter's 1.25 s, and was left to this notebook.

**Measured at the phase close (Part A above), on a clean run into a temporary store:** 53.2 s in all: 8.0 s reading the 124 pages and chunking them, then 45.0 s for the 35 embedding requests, 1.29 s each, against the limiter's floor of 1.25 s. Replies took 1.41 s at the median and 2.91 s at the slowest, so on average 1.8 requests were waiting when the next was sent, fewer than the three allowed; no retries. **The pace is the limiter's: no unexplained time is left.** The earlier 2 s a request did not recur. Those runs came in the same stretch as the DNS failures (entry 6), so slower replies then are the likely reason, though it cannot be shown now. Nothing changes: a faster ingestion would need a higher rate than 80% of the provider's 1 request a second, against D-43's rule.

### 6. DNS failures (Phase 4, concept `document-ingestion`)

**What happened.** Requests failed with `getaddrinfo` errors, and `git push` could not resolve GitHub.

**Decision.** The gateway's transport retry recovered the requests; the computer's DNS was switched to 1.1.1.1 and 8.8.8.8 for the pushes (`TROUBLESHOOTING.md`).

### 7. A new package could not be downloaded (Phase 4, concept `self-rag`, step 1; D-54)

**What happened.** `uv add rank-bm25` failed: PyPI was unreachable from the user's connection.

**Options.** Wait for a better connection; install by hand outside `uv`; prepare the dependency change on the assistant's side.

**Decision.** `pyproject.toml` and `uv.lock` are generated with the newest `uv` on the assistant's side and placed with `take.ps1`, checked with `git diff --stat`; `uv sync` then needs one download per new package. A first lock made with an older `uv` had the wrong format and was regenerated.

### 8. Hybrid retrieval made the answer worse (Phase 4, concept `self-rag`, step 1; D-53)

**What happened.** The Walmart answer (page 23) ranked 10th by meaning alone, and 19th after adding keyword search. BM25L had been chosen on a four-chunk test; on the real report it ranked the answer 108th.

**Cause.** BM25L rewards a word repeated in a long chunk; the question's own wording ("How much ... comes from") also hurt both searches.

**Decision.** BM25Plus, keeping only chunks that hold a query word, and the sub-question reworded before searching: rank 1 on a hand-written query. An acceptance check that passed for vector search alone was rewritten to test the answer chunk's rank, and a wrong claim from concept `document-ingestion` ("none mentions Walmart", judged from 110-character previews) was corrected openly.

### 9. The provider cut the models' limits (Phase 4, concept `self-rag`, step 4; D-55)

**What happened.** The first live run of the subgraph failed with `429 Rate limit exceeded`; the retry's 2 s and 4 s waits could never succeed.

**Cause.** Mistral had cut the Free plan's Ministral models to 2 requests a minute (from 3.13 a second), read from the reply headers. A first fix chosen from the console's Limits page failed live: one model allowed 0 requests a minute, the other is not on the plan; `ENV_AND_CONFIG.md` already listed the first as not to use, and was not read first.

**Options.** Stay at 2 a minute; spread calls over several models; another provider; a paid tier (ruled out by D-41); fewer calls; a model that still allows more.

**Decision.** Probing every chat model the key can see found one, Voxtral Small, at 60 a minute: both tiers moved to it, pinned by its dated name; a 429 now waits for the next minute; every retry logs the provider's limit headers. Limits are read from replies, never from the console page.

### 10. Self-RAG said "not found" with the answer in the document (Phase 4, concept `self-rag`, step 4; D-56)

**What happened.** On the new model the Walmart question ended `not_found`. The first failure printed only its end state, so every node was made to log its decision; the logs showed a retrieval miss: P-15's search queries ranked the answer below 20.

**Cause.** D-53's rank 1 came from a hand-written query; P-15 itself had never been measured live, and the new model ignored its rules on how to write a query.

**Options.** HyDE alone; HyDE and the question merged; a stricter query wording (measured, failed); grading more chunks; a re-ranker.

**Decision.** P-15 v2 writes a passage shaped like the answer, searched together with the question in one request (rank 3 merged; the passage alone ranked 1). Checked on two more labelled questions from other parts of the report, one worded unlike the report, so the fix is not tuned to one question: ranks 2, 3, and 1, all answered.

### 11. What the tests could and could not catch (Phase 4, concept `subgraph-wrapper`)

**What happened.** One planted bug was first missed (ignoring the revised path, whose only visible effect was a warning); one cannot be caught by any test (clearing quotes that every later path overwrites); no realistic leak could be planted for TC-DOC-07, since the design gives document text no path to the web.

**Decision.** The first test now also asserts the warning; the other two are said openly in the concept's notes rather than covered by tests that cannot fail.

### 12. The same web article cited twice (Phase 4, concept `subgraph-wrapper`; D-59)

**What happened.** In the live run with the annual report, one Yahoo article came back plain and with `?fr=sycsrp_catchall`, and became two findings and two source lines.

**Options.** Keep every hit; drop every query string (would merge different articles); compare pages after reading; the page's declared canonical address (needs a server change; parked).

**Decision.** At the user's choice, fixed now: hits are compared without known tracking parameters, and each page is read once.

### 13. Smaller things

- **Phase 4, concept `self-rag`, step 4:** a PSReadLine crash while pasting a long block of commands; commands are now given in shorter blocks.
- **Phase 4, concept `subgraph-wrapper`:** `.env` had no `VRE_MAX_MODEL_CALLS_PER_RUN` line, so the command meant to raise it changed nothing and the new default applied.